# Download recent NYC yellow-taxi trips and export cleaned observations

**Share this notebook file by itself. No repository clone is required.** It contains a snapshot of the project's tested preprocessing helpers and writes those sources into the export for reproducibility.

**Local Jupyter:** open this notebook with a Python 3.10+ kernel, edit the settings below, then run all cells in order. If you need Jupyter, install it in your own environment with `python -m pip install notebook` and launch `jupyter notebook`.

**Google Colab:** upload this `.ipynb`, choose a Python runtime, edit the settings, and run all cells. Set `USE_GOOGLE_DRIVE=True` to persist results in your own Drive. Otherwise `/content` results disappear when the runtime resets; download the ZIP before disconnecting.

The default selects the newest yellow-taxi month linked on the [official TLC catalog](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). TLC publishes monthly files with a lag, so this is recent **historical** data. Select one month first to limit download size.

The export contains `taxi_trips.parquet` (all selected months combined) and, when enabled,
`traffic_observations.parquet` (all selected traffic days combined). Taxi columns include pickup/dropoff
datetimes, pickup/dropoff zone IDs, distance, fare, tip, tolls, total and payment type. Traffic columns include link ID,
observation timestamp, speed and segment travel time. Analytical metrics and summary tables are not calculated.

Traffic is kept separate; no traffic-to-trip or traffic-to-zone mapping is performed.
`payment_type` is retained so credit-card tip propensity can be calculated downstream.

## 1. Settings

Leave `MONTHS="latest"` to discover published files each run. To reproduce a particular period, use a list such as `MONTHS=["2026-09"]` or `["2026-08", "2026-09"]`. The selected month(s), URLs, cleaning rules and source paths are recorded in the output.

For existing downloads or offline runs, set explicit months and point `EXISTING_TAXI_DIR` to the folder containing `yellow_tripdata_YYYY-MM.parquet`. Missing selected files will otherwise be downloaded.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

MONTHS = ["2026-09", "2026-08"]                     # Or an explicit list: ["2026-09"]
RECENT_MONTH_COUNT = 1                 # Used only with "latest"
DATA_DIRECTORY = Path.cwd() / "taxi_clean_data"
USE_GOOGLE_DRIVE = False               # Colab only; invokes the normal Drive mount flow
GOOGLE_DRIVE_DIRECTORY = "nyc_taxi_clean_data"

EXISTING_TAXI_DIR = None               # Example: Path("/content/drive/MyDrive/nyc_tlc_data")

TRAFFIC_MODE = "day"                   # "skip", "local", "day", or "days"
TRAFFIC_FILE = None                    # CSV/Parquet path, required for "local"
TRAFFIC_DAY = (datetime.now(ZoneInfo("America/New_York")).date() - timedelta(days=1)).isoformat()
# Defaults to yesterday in NYC; replace with "YYYY-MM-DD" for a fixed date.

TRAFFIC_DAYS = []                     # For "days": ["2026-09-01", "2026-09-02"]
INSTALL_DEPENDENCIES = True            # False if the required versions are already installed
CREATE_ZIP = True
DOWNLOAD_ZIP_IN_COLAB = False          # True to trigger a browser download after ZIP creation

## 2. Install core dependencies and choose storage

Installs into the notebook's current Python kernel. Existing compatible packages are reused. If the environment requests a kernel restart after installation, restart and run again. No GeoPandas, Shapely, routing service or notebook-server package is installed by this cell.

In [ ]:
import sys
import subprocess

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "pandas>=2.0,<4", "numpy>=1.24,<3", "pyarrow>=12,<24", "requests>=2.31,<3",
    ])

import json
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import requests
from IPython.display import display, FileLink

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Drive mounting is available in Colab. Set USE_GOOGLE_DRIVE=False locally.") from exc
    drive.mount("/content/drive")
    DATA_DIRECTORY = Path("/content/drive/MyDrive") / GOOGLE_DRIVE_DIRECTORY

DATA_DIRECTORY = Path(DATA_DIRECTORY).expanduser().resolve()
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Data directory:", DATA_DIRECTORY)
print("Versions:", "pandas", pd.__version__, "numpy", np.__version__)

Data directory: /Users/Katherine/Downloads/taxi_clean_data
Versions: pandas 2.3.3 numpy 1.26.4


## 3. Load the bundled preprocessing helpers

Run this cell unchanged. The embedded ZIP contains readable Python source files, not data. It is checked against its SHA-256 and extracted to a temporary directory. A copy will be included under `code/` in the final handoff. The notebook uses the bundled cleaning helpers rather than maintaining an independent implementation.

In [ ]:
import base64
import hashlib
import tempfile
import zipfile
from io import BytesIO

SOURCE_ZIP_BASE64 = 'UEsDBBQAAAAIAAAAIVA0q0ZrTAAAAE4AAAAZAAAAdGF4aV9waXBlbGluZS9fX2luaXRfXy5weQXBMQqAMAwF0N1TfHIAwdnRXVwdo01poLYljai39z0i2hJ3wYR1X+D8KrgEuHGMeqKZNDZ2rWVGfYpYT9qgHb1ldRwfrhruLCMRDT9QSwMEFAAAAAgAAAAhUNJlKfaEBwAA6hMAABwAAAB0YXhpX3BpcGVsaW5lL2FjcXVpc2l0aW9uLnB5zVhtb9u2Fv7uX0FoBSzhOnKydRuWzhfITZalwJAVW1qgKwKBliibs0xqJBXHDfzf73OoF0tOtqb7MMxIG0siD8/Lc57nKEEQXL19f8p4+kclrXRSK5ZXKqUvlhmR89RpIzKWG71mbimYE3w9tkwbuZCKF0xpJ+Zar2wcBMHIL8u4E06uBZPrUhvnryfd3Qmj/zNROF4v/6iVkCrX7fLfcP0a1/XTkrtlIeftwze4HDXfK1PgSWzEH5WwbtTeLrnKuGX4KbP2XrPGjkb/O/v1h+TtLz+xGQuWzpX2dDrNvvo2ld/cfazMNl3ex2mhqwyHKxcr4abOyPII3vNgdPPL2eXl6/NH+/EwTqXb6lyJzVabVVzZqRFWVyYVU/lyIY/c7/Pv4t+tVsHot5+vP9OFtbRpMBqNMpGzJNMbVWiehYgfaUVYKAQVLDodMXyCpqZGVFYwcS9pwYLlshDIispYWc0LaZcMvrLWGCqqii3juROGpXpdFoJM+qKS0d4xcJuqEPZPPlwTl9wIOL9eZdKE9YWd3ZgK5fceJXrlL+udMiccDQz4VTZsYqKPE1RLbrZwoL9yI90ysVWey/u+T3F9i/2HBeSNC6K9JbPdm6XPEEkxLo1A2cWdqJPcHR0NtnW3sbMseCoe54Q+OTVKcXDkfm+lCqlWIWpsUadhYuBGZVQ/3AYFZVUUiSvScCu4mbA1kLKcMF25BAmfBYTIqeGbKZYE8H9bCjsLg60oCr0JJtEBVFoUAASC3fx0Xtt7RcdzqdjCCKGml1fv6N/VOw+iCyPvhO9OZquSmmyPFXT60K+TiLEv2B0vJD1jc5GDVNiarwiYXG3b/vTbyaZFjT/c+kssZdSCCQXB4I0PZp9MYGf/mFCEJQ9dpCzwztOXfHnX/MKX3bAahkv0yjteVOIHY7QJ8+CtWimkxaeDbJ+yh+6cXR9M3CyEa5uiqUDEpjhJm2QpjQiGPs5mrRNMFDh1sK8zqzgIdMbyoHdqQl+ptMkDpXd39ODze3r8ZbYjjCOHLugs+DzGvCyFysI9a8Biy4G76QMdsyOE1FFM/bnRAHzeTgM7I3jmYdc5NWGDSnvvnoJgqotqrezsGhB7RFRAHmeE/kJ0WITotKguCfjUQsyKQqSezhp7HehyXWSgrpabGjc+uw7dRm80BdI9ZD0emyOmf6MokwHaGlp4lrnkueYexf35Xt52/Ydwlbh3YVj6/iupqXrJQB7LjqGjSe/Rh+PbIXiy2GOmOSIk43swNL8joOsLdg7VAcXNZQElZcTqfuDoJo0ftSZ81MTTjh3s7M3reNTCMllk/umsA+qALg2HIqSJLYXILHK19WicIOKFmH19jM8ECAMNa/UUTnPh0iWAio2YgC5+vmF6boW54/W0VBGAhy7DMNE2oAw3WaF1GY+8yQtMVlXhLHMa/WMhujAaszeYGWCQdLoo/JxkHV+X9hW6AjdrjcyqspAp8Sjpho1HdfUtNnVzzlHaJBMJayJioYgXcZ1Xrhh44cjplVBsiVQJIIZEGIkRWdxG3Woz5Yd9P2PHe9J8RJiBX7SurAO9s1LTMHknGp5EcCTa8Dl2GhdhxI72YyBVws5OvAe0UlpG2a+7EjcOTdBcKK0GMNfc0Wa/E+OrQrYUqTDfopAOMQ1MbJbCEDZAgb4TuE10zv47Y+MHLNndHB+f+p8xO7u+YMGwxwZ7vq+3YLZ4FETPTBO80ZuenKWFxCyE67YsaK9uOq29xKDGaAToZVvYEhAj5+v9Mcg67E2jhGHD13b2MPA6eOFjDk7r2KF+L7RBsXGjF87EAymR2YTJ7IBbgheFXEuHDVRfbyDPraAbhVAhxRbt6pEeEjY7+fI4euR17MGSEA2jQK4CZ3Rr5px6arZfSyNy73kzGfaK63eQy9ZFnxDxgHqUOq8mI7QhgF+B2EqICK7aQ3tqTgGB2Rxppj9p4AqFXN8FBCghQwfmYJ3VAftdIM2XKI2oc9XwkX/fMiKhdx8bHoxtIketEGpwwD/NJqaJxSTohUYTkMqqKtn5r/VY9lFC7zNmqfmJgHSxXeg/lcnB1NH4PBwV2neV3dTxe+n9Teoj49TSPNWpWPDkgmgIp0/ZtjECeNpq/SiK+oxuRIpmaIk9pHPnPF0la6kqiNHs5OsJ4LN5Ns1fNgw/15XKPEDIPrt+f842YOSj+q1WOSL94hVeioVdwjNLlJUuRbqi1AtqRSeKbcP1b3EWu6pIM/Z6QMgExkoNazG7aNkd78EI2y25I6he3dy8wXSdpnTEWnBVM4Q/1k9aRB5+caVqKSKf/dsc3rhwM9VqQerymNLpsGOA+DBlRPMnL18ek+Xn0P74kYFWAjAxhEg0GbutXzufEohxM2hVxtS0iHLR2e1fC2Jch+1fBMLx2VoYaN/0WmyS93jPHkfdC2RjInYf6Y8Je5f3ptsV3JJlwtRfW+7e6mqTNVa8XVCR6VkciFmLvcPERJ/SIG803gtbtHuGENHGNq7B1p74TPai8+F28o8Iz7gWnnEnPONaeHBj/CfCM54cWPDCM26FZ9wIz/hp4fnq36Q742fqzvgf1J3/A1BLAwQUAAAACAAAACFQa/P8Lb8LAACvIQAAIAAAAHRheGlfcGlwZWxpbmUvcmF3X3ByZXBhcmF0aW9uLnB5pVlrb9w2Fv3uX8HVAq20HSt2+kB3UhXIxk03QJIGSbqLwjsQaIkz1lpDKaLksdPtf99zL0k9RjNOixoGZkSRl/d57iEnCIJ//vzLUjRyJz50sizae9GoumraQm8W4uKn90JXzRYvPsq2qPRCmBpfZCm2sq4xR0idi7aR63WRidtC7UwcBMHJuqm2AjOvy+JKFFsSKN7g8cR9b9SJ/6q7bX0vpBG69kM1hGIA/3Xej93Lpql2cS2bD51q+eWHk5P3b58+f/7iWfrsp5c/v3r9TiTiMigLfZMWebAQgamV4i9Q8VaVaVtsFT3mspWpNGm1pqerqqm6zXWwOBHzPytOS7uQH+qq0K0JVicnJ7lak/PSVt4VqfNgSIabhai6tu7ahDd7ZL1qHl139ynv7ibHmbkNoiXvHLhomExqDskbZ2yhxZVss2tlnohtYQw5PqvKbqsNPjvN7vAvbmXZKRsGEuo2Js+s3MCHrmhUzr568/PLKuPQvrgg+y5+mj63TVGneWFaqTN2QFu1skzllnYNrMB11XCsSU02fdn70UcrQbBiZ83zolTsoqifhsBiyq/BGq/Y00vOFjsrphHsjEkpG4u3Tm68Va0kb8bIohQTzCyEgfVTvxIbhH61gUe3yANOLNrFRL/163mBashxv64D59z01+w3CDljmzMy2HtzvjDuaqimwo+qqVKtNvDqrUonDk3OFmJ4M/JscjY4R97KopRXpaKAZfOdRbG2A8fNWvXCaDWn0nhBAXVTl2Chy6uk33ZhF6Sm+KiS8zP6i5YTN+drqMaT4rZKbfWG0WTKTOvlrNZs3lKq5CQGEVVNkYX5+jJbLYSCNY1JEE7VIBOj3mpsrkqjaNU7LIAFuobVgKpC5+ouydcxf4lmG/pAXU7DuxJfJFjbhq6QCqNlGMWm24bRXAirkST7lTI3b7Jj8EBSzBRAuZz9jv3Hhfmp7Q8m3Xzj9sDGlFQus7280UsLNjGag9J5iLmRwxzTla2N7QXq9XmDtAzdbDuFC95CZkQorzTK+yYvmtA+mOR90yEZ1R38lFY3/DiWTkkDMHUyfPifSySHn9Z2jXazHXb77kYx4CaGfNtDY7dMCoSHqpCaoi2xhaghTDW33AjLUnSGZ8AuBf/korqitwynJj5hqa+pHgUsEpk06rTQRmlTUCxi8dqmvOCu9WjUskgHiVznrasOBSA6XcDPLPKVA/6qpo3Qm9eFKnMjrlRWYa3uyjIWb23dGUEdTLy4WAgSDJO29YKbuGyBjdK0LLLSWFdpnyaQiO2UANxCPyjqx7G+7agH5QoNb3ettHWJhv/Jxk5bl1hF3/sdrQNe//JMlOg2pdiR81ifBYgFcITMaa+tr9dlJVse8MvF0zcvYh8j/nROSYAGccMtpMexUm6vcikytNW2CTPkM1UbCrqsdqpBZsdZVd87uEI1WVGxWx7nXV0WGUUTS6TGxKG2GlkAeP5FxfIDwVMYXPjZPSfybVquUSkc8ymjCvYKxG3fKE7f3oo9njNR2q4deM8KYlxJuKEYcb2vVRiQ7XoTsBN6RyBdS5nhLfo76vP106hv64ywl4dp1GpwxBHodoodgu8RYrg5WO1gB2nUKELxwti8c3gUic/8lA1BYtSbxSnyzVfB1B8jkrfqdSPgIvWdcvEwaabkglyAdEzQHu5g/8wtPW98iCQuD5raf5+Fhqc7IElEuBdIXbXUjeCJmf7+3R70+5mXx8K48gs5vcFjTXIe9eUQ2BIPyGKbm4NBrvr3w+5S2L59IPTOyM8SJ8h1WvE//8xRnkM3Kje7tItXMaFNm9piyZuqdm3BonuJYKNZeGh3n3v4ru6oZou2pKNP3mVDgarhaNNWAnjrBI5xXdQoa4rOiHCzruG8tfj9Y4O2l9pUDqdnkRtYkgS20wTTUMZkXtrjEVYOB50bpeokKJFLQXTEJd4n6KZGpaMUtdW15xQC3t01ag/nvUydGoVl3NLggaLtcrUoK73hb5BYgCaffi/+/eO7b78SLwut3nEyu5b3SpZUSCrnI9FptT5tpN4o23ZytVFakWyxURU3GCCJ8+Fr9KEp0PPB0lzLWpX3sVtw70+Yw84+e5HZokBztMzKWrqgRjBG8WEvHoPA8QGm4CPAZQu/qxCn3pDBBkgZG0SiDYNFEEWRPQRZguteNMF/zBeIDW/qwTYaqDjUs4cR7BCJ78RjARFUgjwaib8kdohsGINhjUIty1mdz//c2vD072fiOxTp5dmKPvFEjj89/9YNn/MwHj8tsrfSaj2lmfuOpD/kGcHYEJnwMizpHgF5ZJ1WkjMx1EtdRfuhYSHkLnyCDesNzprf4xDGvL/fTt1lqoa57wGn3I4Xo9Z8JN62JK66osx9ldrS+Ii3qbvj8HW7EDQ6HOxJwl7ZYIVQEqcrIvOFzlpLt7ZFzrVGMCJZyhNP6TpNu1BhYaInibzPUvyoqp4si10Bq290tdPi2dt3HMLhqP6IVjxy/chzDs8O7d4GeKvED2/e/bh8/PibL0VI9GutVBs9Ic4PGq8yTILSJYrasA7iCgw/l1SSVtg/7PM9uBimZNdVBZkGMFcSLA76xOKlRUpGG4b5vlh3hVftLUfCiNA5egHNb4EFGwX/ZTgGiHdoBEBff9lUGGHQuUCsNdCa6SadJ7GRyqc44RABmw6XSZs697jADo4zlDZEUhwfInXv0RNP2R29DdsOtl1DFYQT0XBIzVKR7fyJfju9SqHRyW2TCLw8asIjQmfXj5evZk3WWjBMGfosGxAM5AjB7alRz5NdSv8pstwLO8JZJzdxY040vYBbjaVNWawT/GdY7L6S1BDBMkyHFt6OVHRaFLnp516O3q5m3Xeq9cNc0+/9R7nmX8XbSuZD2tmDpcHRiFMIOIT8y+X90tbZIejxa508LpwKvXDhAIjLCiK3laFr2QxHtwm9IYju9zeqBEhgOJ76tqd8R3mNlgg+nXMrnQRroHwb7HebmYeP5s9qSnimkRhqai+B7OKYWviMA1kR0LzIR+bs58og+iEK92lTfy+T60k4MQJWbtTGqBlSrgLV4nGjsBMvJ84b4QwwltKO+8BXXz7+BlVE9yaNCakrRFPxe97ksZ51xdw16oo4cXgWf70Y8ebR5Qz9/RdO5gtnUtbQU8iyXEedaLAQ19UOHlDrNuALlpydlARDkxr5k28LWHRoN5kGZgSh0XGvH+Q9Uw8+DOWr1Vgj26sSwpIYYL0h2shagjkgJQapY0M5amzn+fLcaUT8ZnlEsDSm2Ohw0Cth0LMe9d+dfskID52UWXdx46OO0sPti1EDcWTFXhMeWOOPkqNLw76n0w2/vW9kruMu450QxD2wwvu39nH+o0zgOdNBOeL0+EK3rCbI5V8Dzs/OxN+8UY+mclzd9c/MNs/is998VVoaOAToT9xgjiJjr9D5Yv6TV5l+yeET30CnHM/dqP4snF7dM8cduG3PwdagVIe4rbv+ZCJkLwINdwfHyBpaaK755Mg/yVgqe6HWEmf25UNH5wXd9Wmh6Cex050qNteueUmrvqOLrNjn519vC/350r+zPY9nE0Mu6KJ2uFjky9OrLrtRrdvELTtxGHdj7LboiqQNGReLpzlYgHcU6Gpe7WLxjC67zdBd2beetz9xJ0e7ZGgHRC/pHlOrHRmfVRp+u+r4OnPkiQUf1NaKfpchladcdnQdkRy4sPa3Cj4tyU1zWjsRsncvMrw7hDvD28tg6hQGjuH1+C4qb2Ock8GeSZvosBajlRvgVH11P235+5stQOLTcSnEcrMJJ1XON1xJONx0bZXUASphdOOF13u/BrtJ09+0eluSvf4OFLnzvblRaySBZhLkkfVY35hy/H7l/r2tG/5dpNeHfL7qj11bv3KthY81lKR0MlV3MmuZLSr+gdZX69Ax+7Y+iqRteL1CB1teobVqJj1v2/e8G3XP9y3T/vSFCA8kn092RubL1QQaHRvwiUViP5E/8naTHs0fevmHcmhym8A/lCVjmhfoThfAeJebe7BxLOX+D1BLAwQUAAAACAAAACFQzPmFoH0MAAAWIwAAIAAAAHNjcmlwdHMvcHJlcGFyZV9jbGVhbl9oYW5kb2ZmLnB5nVrrctu4Ff7vp0CT2SWVlenLpvvDqXYmjZNtppvEs8m23XE8CkRCEhLeDIC+rMczfYe+YZ+k3zkAKVKSHW81SWyBwMG5fufCPHr06OVVXRkn3GUlqplV5kI6XZXCyVmurLjUblk1TqQyT5scj8qFkKXMr53GkiiUMzq1yaNHj3Z0wYQ+26rcmZuqELV0y1zPRHhwgq/tJqN22l/LpqivhbSirNulWpYZFvCnzrq1a2lMdcmLcm0xqaU5b5Tjh+c7/nYnr/S01rXKdakSIy+ntVHY6MVrL69MIXP9u5o6I+dzne7sfHj+r9fTF+9+/vXN2/diIk4jV6sahNIvTT3NpFNOFyoaC7+emaqu5vPBg5Nff65Svub1cTTeEWuf6Phd/zkIGQ1C2jpZpkxgLo2ayqJqSsfP8bj9tknNVXlu+7srJ/Pe91peF6p0U3ddq+jMi/f+xd9evnkO6WqZ2HSpChmfMuU4HdMaSQJ2ijqOShuNhJ6LNFFlZskd4qgTdiRUbpXnCcd06X54Gm9s70k7EpURqZhM1vhiOkRinleSiYzEnLfqUvQtsnM22tnZydRcXMBuxMiUDB2Tr42Odjwj3hkg3Xly4r+80rnye3hLoa0lT54Iq1zcpz8Su7wWiATtTL2blRJq8RQgYksEbLa7EQ4SPMkETj3FCUuS7h91RjNSQ8x/yLxRL0HRxPPodcmCsLuKOdgUN8Tn7VFLf3Jj4akqi8P30e1YEOnJzZ233kaeSaNcY8q7uQuaNE0Z+2jBxXYsQijw17FA9NeNw/7KjUNQmepCleSsk7dVqYLaaQMUSlEe986MuodJ8SXThjQLs9vJB9OosVBX8Ptp9YW/+r0xU9oTEcIVksP/wskH7N2DLNNClnoO/00IinC6KQEBX1r9DQl4TsF3S4glpIi0LapEvNEpQgxprrHXH0rIu6e2ga6u4qjdTT9d0P95A9M6OnF61npNiWtWyr7HNaL3KlcpMM2JXEnrBFTtvaSoSreMOj/MVRnfsNrrEbsoR05NkbO66HYk/jThrau10X23Hzd1Doh3/TttuNSZ69VR0kIv0v5ptFMm7tQ1Fj3EGYu0KgDDMERVTiJbyrq+BigAty/53NEA31gOcEqiFLKOT9ghtwrQfragwsaeQrp0SSZXybzJc/4am+ha5Xl1ybanKJnGH7Obp7e7+PfwdvSx84Yxs8SK3iQdDMwkN5m7AwKQgWFolYl1Dn7DZ/fNm/bqowAN0RaRyDyEeFlyooyugBbEwunB2VjMjTqfRG+2nEopRwCjxE1EyBMdrURD4qCUSSiB5X185axvte3Wbjfokb1mrFwYbBv4JmTjKW9RNuafU4vsOzncpw+5R94UJcChD8nbFTk35OoTf2HiqqmvGuJNMVeinq6EOhPf+XhgOtsPcQZijtbT0OnR4dl2tjrWTv3JM28V8NdmzXjwGBBIbmAnUVopw/l/TkWJm0SFvlLZFrN9hbXDoz/MGhICvCb9CmfbOcmaUFJNhD+fbC2MkFd7T9fKqVGSucQXLlalFSoHVBF74of9rTf6hInr2quTmXKXCrY8GIvDp/vb+ew+397DSLJQMcdSgvLHoDjRd/nGw8jlLvb0xHfiYPR/EO2XhsnCxfujO54hxA7290cP8ZbTtQp1rSK9x3u85r+dDN1ooP4fvu847J4XwCNIr87jO2xzrzfvHjyEo7JOtJ3rEggzdOM/fOOfH3jjUEY4zh3CPRZ/V6oWXFP4bNlWjqEEFlQCj8UMdYi6SvMmQ85F1Sm4jUHPpcwddMP5XYv8oVGx+c7MKpuIV0amZE+Z75VV6fUCgTO0dDNEGOKxROpLthJu2WpFXG8htp1pLPWL09XRuG+QsEyeEX79uk9sflaHW4dDxjh88uT7A4DLwV2uv00E7nw8qculMioesj9KUG9hY4zqHM3IXRAcEpCPxrxKT9k1EqTSvJTxK4mWZjQe+NZ23fnSJ+EfU+67obDkA/2SkA+0mY0vGgvfkkwGZRXVVGjd1VSXmbqa+Lvvz4TDfP61dNjlT1Wg+M/aQ5NtiXX3rjs2qIYCOUEZiHYx9sdWl4dqyjZFfL7JMIXwOUVvoLJWJ2yWtG8rX8v6SIQkEoflHLoXKUrsEiHZM3VXwSboLXKZqtDV+B1wbJnnvTJ4tf2+diNMVdAQeGJnoZ72DZdvYrge0yUDQYZCi40JkM71hep9/R39AFqduqZL5IXUOdFuN9wOCG92Oe2D1cRnrd+B7vt9oNCWbUEN30pofhJ6vv7ulRbhFr7KMEpm09RexL6rzCi+JjcRqWqqSczIwi6wwC3PD7gQ9b0VQusSHcWIpwYJSLTTgkA0sL1W7Ad2cPnGjCcGUwMnC+sJTOiuH+BFHi26K/pKfLhbgbOheb7eT3q51jhsmecK2Gui13z18WB4bMO9h9ysdm+4+sPdfeXybYjfdckwBJKGGiAVt1EQhgUtxkwIqMiI45Vxs9WTQGu0OSwbfpA2cQzpkVLhRJeuPZnwkwT+DhxH1Qb8GX2VGs5eqJyLu600e8//IOUGORRV+PG7D8JWDSpxv/KMA5IMTJpF2sqvxQXK+LnuugaKk6MNHQd3u89wurwAwcr0Zhf9ZtzbtI8DXx25Dai23rC97/T95Z1Tq/vUFYUGsnd+c4KHO2bXTtn2XlTQjozhuBe9bWdnHRbT84Z2RzS9yJXz418aQ4TUg2fht3GHrVgbOvU639HaMI0ODFf4FtI0HnW62yAD+dIvtOXGUyQdTdFEcUGQ9Y+e7p+dRm3mnjw4r97ePnzWFooYdeViWkmypqhtHBTAzlq6yeEI7VD0sYzWhni/vHx+/OZlUmRDQp3A0WPxghDVp3EURVsh+GMJyqsznzYnep+OeJgGa850qTLvziw3gE5YnruprD/4SkSP4jujF4SJ4sPPL9omgh3rmegG51ZIo8Tb314IFIbYekmU+SlyX+jZKKcW8H8s4Ra4OcLNz+396V/fHw/ufSHtEjQCbQp+g1LeAB8T0a9via4BOZaN3qNwrc8yApDUQplnfbK9OTaglIYPOOZbhZDM6Hj3Jgjcp9JkfAJtirmkBBlm2KzkPm0ky823RXSTXCyMWgDjvTCKX0JBkDXjvQgJNMhjhW+wLYPQUmnTIiKbaSyoKBKvj604+O+//4NedNyNJ8LS032ovGwco8Dqns4kP+6zX/1lgma6NQ4thA6KgJ3YRikm6C3NmOyBf+gdjHdIGmK0Vhyo4sMSJTqKd4PGxXaNnlHzpsz82ZX2KRLXVfHpvprtE9m8zZYCPU0JynhgodMjEYosKIOmitJOq/nYZ7+BEnoZivnpikomIDogFu97WYjN13lb6yS9rBRykh8vDDTyC70gITY1SEi+ZLyKIM9Cf/gdyi7vRc+E0uR+4atIAQsz1bbWQ80HkAgl7OfKB/2gG4BLku8Ea6dAYCMXqj2yaKSRCJ1NB338WBxXlyVqVyWL1VvSTRj6gGtWs7I4OOGIx1h+VIa+yXv3iLw7zMF4CjbU2skb4bG7/6aPxmW01t4w7S5gWFuBGkm5xya+hJGqywEUuCW5ESHGwlSWjJMqXTu0l9XV9ZhVAZyir3PtBlwh6Sto1KdKnvJjwXqdCtImhauw1DHXPKAeUxzqQv/Ob5SDuqs0bWqtPJKstNW7x8fJVXsQykjEcx/24LtiVKDBSek07uRAKCsPVN2uzqmAjYuqyvr0QWGG3Ugon6EvhLkPTepNUiqwZspRYY9tfddcc7c+71mlvA8V2NwYHu2wk5MDZoaABL/IYHrmcd3H3u0ek8pncqb5dVL8DbkNAdSTlV2f77pq969ecdxHwyGG2Yz9nGdKz4f84gTII1ytf1mlcuQ3ijRgVabdLsF9f3y1eh8tfhT7uOeeA7174kGOQv1xMHrWllmkFIBFBZ+QjhwWOm8hsim/lAixwbRsCK2vcAJMiR8P97/pi4Kup+oApvdSndmmKxqr+tLsrW9KDvfXjRHAZFdmnxtCV7BdEKkS4ECBE2ZxeEBYDjAY+7dB5K3U09h+8HnnGkIwU4ct2VtDfy9iBkMxk5b/OwM3rUvVXkFyUNCVC0AnnG400M5P+kngjkkuVEWQeb2XQpmm0iH5WJTCmgoVpRdLSl0n7JB7LTjpYiZzTpGEtFxbDuKSZM/Qji/xt8ULmmheqKVGG0xJeQ/SUVRVRqs2b7BtMg9SqUPBKbgJ58jYwHGC67YAgOPQzKtcFUHMlAaDNPm8IPsjBJeMPOCYYqGWFidQ/wSjJV0JGt6Ue2/c+R9QSwMEFAAAAAAAAAAhUAAAAAAAAAAAAAAAABMAAABzY3JpcHRzL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhUDSrRmtMAAAATgAAABkAAAAAAAAAAAAAAIABAAAAAHRheGlfcGlwZWxpbmUvX19pbml0X18ucHlQSwECFAMUAAAACAAAACFQ0mUp9oQHAADqEwAAHAAAAAAAAAAAAAAAgAGDAAAAdGF4aV9waXBlbGluZS9hY3F1aXNpdGlvbi5weVBLAQIUAxQAAAAIAAAAIVBr8/wtvwsAAK8hAAAgAAAAAAAAAAAAAACAAUEIAAB0YXhpX3BpcGVsaW5lL3Jhd19wcmVwYXJhdGlvbi5weVBLAQIUAxQAAAAIAAAAIVDM+YWgfQwAABYjAAAgAAAAAAAAAAAAAACAAT4UAABzY3JpcHRzL3ByZXBhcmVfY2xlYW5faGFuZG9mZi5weVBLAQIUAxQAAAAAAAAAIVAAAAAAAAAAAAAAAAATAAAAAAAAAAAAAACAAfkgAABzY3JpcHRzL19faW5pdF9fLnB5UEsFBgAAAAAFAAUAbgEAACohAAAAAA=='
SOURCE_ZIP_SHA256 = '8bd15efbe9d0dd9822c48b6af1fc8e4a164d02b55d52c6a4c4a3f5e7e4d435e6'
SOURCE_ZIP_BYTES = base64.b64decode(SOURCE_ZIP_BASE64)
assert hashlib.sha256(SOURCE_ZIP_BYTES).hexdigest() == SOURCE_ZIP_SHA256
SUPPORT_DIRECTORY = Path(tempfile.mkdtemp(prefix="nyc-taxi-notebook-"))
with zipfile.ZipFile(BytesIO(SOURCE_ZIP_BYTES)) as archive:
    archive.extractall(SUPPORT_DIRECTORY)
sys.path.insert(0, str(SUPPORT_DIRECTORY))

from taxi_pipeline.acquisition import pull_traffic_speeds
from scripts.prepare_clean_handoff import TAXI_COLUMNS, validate_taxi, run as prepare_handoff

print("Bundled source SHA-256:", SOURCE_ZIP_SHA256)

Bundled source SHA-256: 8bd15efbe9d0dd9822c48b6af1fc8e4a164d02b55d52c6a4c4a3f5e7e4d435e6


## 4. Select published months and download inputs

Downloads stream to `.part` files, and a file is published to the cache only after validation. Existing selected Parquet files are checked for required columns and nonzero row counts. A missing/unpublished month or connection failure stops with an actionable error; there is no fallback to older or fabricated data.

Raw and processed outputs are separate. Reruns reuse cached raw files and replace the handoff for the same month selection; copy the old handoff first if you need to preserve that version.

In [ ]:
import re
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit

CATALOG_URL = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"
TAXI_BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

def latest_published_months(count):
    if not isinstance(count, int) or isinstance(count, bool) or count < 1:
        raise ValueError("RECENT_MONTH_COUNT must be a positive integer")
    class Links(HTMLParser):
        def __init__(self):
            super().__init__()
            self.months = set()
        def handle_starttag(self, tag, attrs):
            if tag != "a":
                return
            href = dict(attrs).get("href", "")
            parsed = urlsplit(urljoin(CATALOG_URL, href))
            match = re.fullmatch(r"/trip-data/yellow_tripdata_(\d{4}-\d{2})\.parquet", parsed.path)
            if parsed.hostname == "d37ci6vzurychx.cloudfront.net" and match:
                self.months.add(match[1])
    response = requests.get(CATALOG_URL, timeout=(15, 60))
    response.raise_for_status()
    parser = Links()
    parser.feed(response.text)
    available = sorted(parser.months)
    if len(available) < count:
        raise RuntimeError("Could not discover enough published months. Open the TLC catalog and set MONTHS explicitly.")
    return available[-count:]

def download_file(url, destination, validator):
    destination = Path(destination)
    if destination.exists():
        validator(destination)
        print("Reuse:", destination)
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    print("Download:", url, flush=True)
    try:
        with requests.get(url, stream=True, timeout=(15, 120)) as response:
            response.raise_for_status()
            with temporary.open("wb") as stream:
                for block in response.iter_content(chunk_size=1024 * 1024):
                    if block:
                        stream.write(block)
        validator(temporary)
        temporary.replace(destination)
    except requests.RequestException as exc:
        raise RuntimeError(f"Download failed for {url}. Check internet access and the selected month, or supply existing files.") from exc
    finally:
        temporary.unlink(missing_ok=True)
    return destination

if MONTHS == "latest":
    selected_months = latest_published_months(RECENT_MONTH_COUNT)
elif isinstance(MONTHS, (list, tuple)) and MONTHS:
    selected_months = []
    for value in MONTHS:
        if not isinstance(value, str) or not re.fullmatch(r"\d{4}-\d{2}", value):
            raise ValueError("Use MONTHS='latest' or a list of YYYY-MM strings")
        pd.Period(value, freq="M")
        selected_months.append(value)
    if len(set(selected_months)) != len(selected_months):
        raise ValueError("Duplicate months are not allowed")
    selected_months.sort()
else:
    raise ValueError("Use MONTHS='latest' or a nonempty list of YYYY-MM strings")

if TRAFFIC_MODE not in {"skip", "local", "day", "days"}:
    raise ValueError("TRAFFIC_MODE must be skip, local, day, or days")
if TRAFFIC_MODE == "local" and (TRAFFIC_FILE is None or not Path(TRAFFIC_FILE).expanduser().is_file()):
    raise ValueError("Set TRAFFIC_FILE to an existing CSV/Parquet for local traffic")
if TRAFFIC_MODE == "day":
    from datetime import date as calendar_date
    if not TRAFFIC_DAY:
        raise ValueError("Set an explicit TRAFFIC_DAY=YYYY-MM-DD for traffic acquisition")
    calendar_date.fromisoformat(TRAFFIC_DAY)

if TRAFFIC_MODE == "days":
    from datetime import date as calendar_date
    if not isinstance(TRAFFIC_DAYS, (list, tuple)) or not TRAFFIC_DAYS:
        raise ValueError("Set TRAFFIC_DAYS to a nonempty list of YYYY-MM-DD strings")
    TRAFFIC_DAYS = sorted({calendar_date.fromisoformat(day).isoformat() for day in TRAFFIC_DAYS})

acquisition_records = []
taxi_paths = []
for month in selected_months:
    filename = f"yellow_tripdata_{month}.parquet"
    existing = Path(EXISTING_TAXI_DIR).expanduser() / filename if EXISTING_TAXI_DIR is not None else None
    url = f"{TAXI_BASE_URL}/{filename}"
    if existing is not None and existing.is_file():
        path, method = existing.resolve(), "user-supplied local file; provenance not independently verified"
    else:
        path = DATA_DIRECTORY / "raw/tlc" / filename
        was_cached = path.exists()
        path = download_file(url, path, validate_taxi)
        method = "cached local file; original acquisition not independently verified" if was_cached else "official TLC download"
    count = validate_taxi(path)
    taxi_paths.append(path)
    acquisition_records.append({"month": month, "path": str(path), "method": method,
                                "official_source_url": url, "raw_rows": count})

OUTPUT_DIRECTORY = DATA_DIRECTORY / ("clean_handoff_" + "_".join(selected_months))
print("Selected months:", selected_months)
display(pd.DataFrame(acquisition_records))
print("Output:", OUTPUT_DIRECTORY)

Download: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-08.parquet
Download: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-09.parquet
Selected months: ['2026-08', '2026-09']


,month,path,method,official_source_url,raw_rows
0,2026-08,/Users/Katherine/Downloads/taxi_clean_data/raw...,official TLC download,https://d37ci6vzurychx.cloudfront.net/trip-dat...,3336716
1,2026-09,/Users/Katherine/Downloads/taxi_clean_data/raw...,official TLC download,https://d37ci6vzurychx.cloudfront.net/trip-dat...,3876044


Output: /Users/Katherine/Downloads/taxi_clean_data/clean_handoff_2026-08_2026-09


## 5. Optional traffic input

Set `TRAFFIC_MODE="days"` and `TRAFFIC_DAYS=["2026-09-01", "2026-09-02"]` for multiple days. The default `day` mode downloads yesterday in NYC. Taxi and traffic dates are selected independently. `skip` makes no DOT request. `local` reuses a saved CSV/Parquet; `day` fetches a selected historical day and stops if no observations are returned. A DOT outage must not be confused with zero traffic; switch to `skip` and rerun to complete the taxi-only product. No traffic records are joined to taxi trips or declared live.

In [ ]:
traffic_path = None
if TRAFFIC_MODE == "local":
    traffic_path = Path(TRAFFIC_FILE).expanduser().resolve()
elif TRAFFIC_MODE in {"day", "days"}:
    traffic_days = [TRAFFIC_DAY] if TRAFFIC_MODE == "day" else TRAFFIC_DAYS
    traffic_files = []
    for day in traffic_days:
        path = DATA_DIRECTORY / "raw/traffic" / f"nyc_traffic_speeds_{day}.parquet"
        if not path.exists():
            observations = pull_traffic_speeds(day)
            if observations.empty:
                raise RuntimeError(f"No DOT observations returned for {day}. Select another day/file.")
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary = path.with_suffix(".parquet.part")
            try:
                observations.to_parquet(temporary, index=False)
                temporary.replace(path)
            finally:
                temporary.unlink(missing_ok=True)
        traffic_files.append(path)
    traffic_path = DATA_DIRECTORY / "raw/traffic/selected_traffic_days.parquet"
    temporary = traffic_path.with_suffix(".parquet.part")
    try:
        pd.concat([pd.read_parquet(path) for path in traffic_files], ignore_index=True).to_parquet(temporary, index=False)
        temporary.replace(traffic_path)
    finally:
        temporary.unlink(missing_ok=True)
    print("Saved traffic input:", traffic_path)
else:
    print("Traffic skipped: only the cleaned taxi table will be exported.")

Saved traffic input: /Users/Katherine/Downloads/taxi_clean_data/raw/traffic/selected_traffic_days.parquet


## 6. Run preprocessing and export the handoff

Taxi cleaning scans 200,000-row batches. Validity rules require IDs 1–263, positive distance up to 100 miles, duration 1–240 minutes and finite nonnegative passenger totals. Pickups outside their file's month are excluded and counted.

Fare, tip, tolls and total must be finite and nonnegative; refunds and malformed rows are excluded.
Missing or malformed `payment_type` values become null; otherwise valid trips are retained.
No duration, speed, earnings, tipping probability or O–D metrics are exported. Cash tips are unobserved.
Traffic retains source speed/travel-time values and units; either value may be missing on a retained row.

In [ ]:
source_description = "; ".join(sorted({item["method"] for item in acquisition_records}))
run_report = prepare_handoff(taxi_paths, traffic_path, OUTPUT_DIRECTORY,
                             taxi_provenance=source_description)
assert run_report["status"] == "complete" and all(run_report["checks"].values())

code_directory = OUTPUT_DIRECTORY / "code"
code_directory.mkdir(parents=True, exist_ok=True)
(code_directory / "preprocessing_source.zip").write_bytes(SOURCE_ZIP_BYTES)
run_settings = {"selected_months": selected_months, "traffic_mode": TRAFFIC_MODE,
                "traffic_day": TRAFFIC_DAY, "traffic_days": TRAFFIC_DAYS,
                "source_zip_sha256": SOURCE_ZIP_SHA256, "acquisition": acquisition_records,
                "versions": {name: __import__(name).__version__ for name in ["pandas", "numpy", "pyarrow", "requests"]}}
(OUTPUT_DIRECTORY / "reports/notebook_settings.json").write_text(json.dumps(run_settings, indent=2) + "\n")
print("All handoff validation checks passed:")
display(pd.Series(run_report["checks"], name="passed").to_frame())
display(pd.DataFrame(run_report["taxi_quality"]))
display(pd.DataFrame(run_report["tables"])[["file", "rows", "bytes"]])

All handoff validation checks passed:


,passed
taxi_rows_conserved,True


,file,raw_rows,analysis_rows,removed_rows
0,yellow_tripdata_2026-08.parquet,3336716,3132106,204610
1,yellow_tripdata_2026-09.parquet,3876044,3635168,240876


,file,rows,bytes
0,taxi_trips.parquet,6767274,147045420
1,traffic_observations.parquet,21095,125636


## 7. Inspect the two observation tables

Selected months are combined in `taxi_trips.parquet`. Traffic appears in `traffic_observations.parquet`
when enabled. Reports and source code document the run; no other data tables are exported.

In [ ]:
for table in run_report["tables"]:
    print(table["file"], "rows:", table["rows"])
    sample = next(pq.ParquetFile(OUTPUT_DIRECTORY / table["file"]).iter_batches(batch_size=5)).to_pandas()
    display(sample)

taxi_trips.parquet rows: 6767274


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount,payment_type
0,2026-08-01 00:03:04,2026-08-01 00:07:51,140,262,0.60,6.5,2.30,0.00,13.80,1
1,2026-08-01 00:38:45,2026-08-01 00:44:02,140,262,0.70,6.5,3.45,0.00,14.95,1
2,2026-08-01 00:54:51,2026-08-01 01:01:07,140,263,1.10,7.9,2.60,0.00,15.50,1
3,2026-08-01 00:27:35,2026-08-01 00:54:27,164,80,5.24,29.6,8.56,7.46,51.37,1
4,2026-08-01 00:39:08,2026-08-01 01:00:23,148,141,3.57,20.5,5.25,0.00,31.50,1


traffic_observations.parquet rows: 21095


,link_id,speed,travel_time,data_as_of,borough,link_name,link_points
0,4329507,37.90,131.0,2026-11-03 00:01:04,Manhattan,LINCOLN TUNNEL W NORTH TUBE NY - NJ,"40.75719,-73.99724 40.76017,-74.00382 40.76185..."
1,4620314,14.29,306.0,2026-11-03 00:01:04,Bronx,MDE S HARLEM RIVER PARK - GWB W AMSTERDAM AVEN...,"40.85526,-73.918591 40.85266,-73.92085 40.8500..."
2,4620330,14.29,311.0,2026-11-03 00:01:04,Bronx,MDE S HARLEM RIVER PARK - GWB W AMSTERDAM AVEN...,"40.85513,-73.91865 40.8526305,-73.920881 40.85..."
3,4620298,87.61,76.0,2026-11-03 00:01:08,Manhattan,GWB E LOWER LEVEL PLAZA - CBE E LOWER LEVEL AM...,"40.8462505,-73.932161 40.846951,-73.933641 40...."
4,4620331,11.18,375.0,2026-11-03 00:01:08,Bronx,CBE W MORRIS AVE - GWB W AMSTERDAM AVE (U/LVL),"40.8465405,-73.932021 40.84611,-73.93075 40.84..."


## 8. Save and share the results

The output folder contains the cleaned taxi table and optional traffic observation table, plus a README, run report and source snapshot. No zone lookup, metric tables or CSV previews are exported. The ZIP includes the readable helper sources and this run's settings.

In local Jupyter, use the displayed links or your file browser. In Colab, a runtime-local file link may not download; use the Files sidebar, your mounted Drive, or set `DOWNLOAD_ZIP_IN_COLAB=True`. Large ZIPs can take time to create/download. A Drive output folder is the easiest way to keep large results.

In [ ]:
import zipfile
print("Handoff folder:", OUTPUT_DIRECTORY)
display(FileLink(str(OUTPUT_DIRECTORY / "README.md")))
for table in run_report["tables"]:
    display(FileLink(str(OUTPUT_DIRECTORY / table["file"])))
archive_path = None
if CREATE_ZIP:
    archive_path = OUTPUT_DIRECTORY.with_suffix(".zip")
    temporary = archive_path.with_suffix(".zip.part")
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
            for path in sorted(OUTPUT_DIRECTORY.rglob("*")):
                if path.is_file() and not path.name.endswith(".part"):
                    archive.write(path, arcname=Path(OUTPUT_DIRECTORY.name) / path.relative_to(OUTPUT_DIRECTORY))
        temporary.replace(archive_path)
    finally:
        temporary.unlink(missing_ok=True)
    print("Shareable ZIP:", archive_path, f"({archive_path.stat().st_size / 1024**2:.1f} MiB)")
    display(FileLink(str(archive_path)))
    if DOWNLOAD_ZIP_IN_COLAB:
        try:
            from google.colab import files
        except ImportError:
            print("Not running in Colab; use the ZIP file link or local file browser.")
        else:
            files.download(str(archive_path))

Handoff folder: /Users/Katherine/Downloads/taxi_clean_data/clean_handoff_2026-08_2026-09


/Users/Katherine/Downloads/taxi_clean_data/clean_handoff_2026-08_2026-09/README.md

/Users/Katherine/Downloads/taxi_clean_data/clean_handoff_2026-08_2026-09/taxi_trips.parquet

/Users/Katherine/Downloads/taxi_clean_data/clean_handoff_2026-08_2026-09/traffic_observations.parquet

Shareable ZIP: /Users/Katherine/Downloads/taxi_clean_data/clean_handoff_2026-08_2026-09.zip (123.8 MiB)


/Users/Katherine/Downloads/taxi_clean_data/clean_handoff_2026-08_2026-09.zip

In [ ]:
taxi_file = run_report["tables"][0]["file"]
traffic_file = run_report["tables"][1]["file"]

trips_df = pq.ParquetFile(OUTPUT_DIRECTORY / taxi_file).read().to_pandas()
traffic_obs_df = pq.ParquetFile(OUTPUT_DIRECTORY / traffic_file).read().to_pandas()

In [ ]:
display(trips_df)
trips_df.dtypes

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount,payment_type
0,2026-08-01 00:03:04,2026-08-01 00:07:51,140,262,0.60,6.50,2.30,0.00,13.80,1
1,2026-08-01 00:38:45,2026-08-01 00:44:02,140,262,0.70,6.50,3.45,0.00,14.95,1
2,2026-08-01 00:54:51,2026-08-01 01:01:07,140,263,1.10,7.90,2.60,0.00,15.50,1
3,2026-08-01 00:27:35,2026-08-01 00:54:27,164,80,5.24,29.60,8.56,7.46,51.37,1
4,2026-08-01 00:39:08,2026-08-01 01:00:23,148,141,3.57,20.50,5.25,0.00,31.50,1
...,...,...,...,...,...,...,...,...,...,...
6767269,2026-09-30 23:27:48,2026-09-30 23:33:52,249,90,1.25,11.42,0.00,0.00,16.17,0
6767270,2026-09-30 23:41:50,2026-09-30 23:56:32,170,74,3.84,20.21,0.00,0.00,24.96,0
6767271,2026-09-30 23:15:30,2026-09-30 23:23:00,234,158,1.19,13.66,0.00,0.00,18.41,0
6767272,2026-09-30 23:56:47,2026-10-01 00:05:06,170,48,1.05,15.40,0.00,0.00,20.15,0


tpep_pickup_datetime     datetime64[ns]
tpep_dropoff_datetime    datetime64[ns]
PULocationID                      int64
DOLocationID                      int64
trip_distance                   float64
fare_amount                     float64
tip_amount                      float64
tolls_amount                    float64
total_amount                    float64
payment_type                      int64
dtype: object

In [ ]:
### Trip Duration
# Dropoff Datetime - Pickup Datetime

trips_df["tpep_pickup_datetime"] = pd.to_datetime(trips_df["tpep_pickup_datetime"], errors="coerce")
trips_df["tpep_dropoff_datetime"] = pd.to_datetime(trips_df["tpep_dropoff_datetime"], errors="coerce")

trips_df["duration"] = trips_df["tpep_dropoff_datetime"] - trips_df["tpep_pickup_datetime"]
trips_df["duration_minutes"] = (trips_df["tpep_dropoff_datetime"] - trips_df["tpep_pickup_datetime"]).dt.total_seconds() / 60

display(trips_df)
trips_df.dtypes

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount,payment_type,duration,duration_minutes
0,2026-08-01 00:03:04,2026-08-01 00:07:51,140,262,0.60,6.50,2.30,0.00,13.80,1,0 days 00:04:47,4.783333
1,2026-08-01 00:38:45,2026-08-01 00:44:02,140,262,0.70,6.50,3.45,0.00,14.95,1,0 days 00:05:17,5.283333
2,2026-08-01 00:54:51,2026-08-01 01:01:07,140,263,1.10,7.90,2.60,0.00,15.50,1,0 days 00:06:16,6.266667
3,2026-08-01 00:27:35,2026-08-01 00:54:27,164,80,5.24,29.60,8.56,7.46,51.37,1,0 days 00:26:52,26.866667
4,2026-08-01 00:39:08,2026-08-01 01:00:23,148,141,3.57,20.50,5.25,0.00,31.50,1,0 days 00:21:15,21.250000
...,...,...,...,...,...,...,...,...,...,...,...,...
6767269,2026-09-30 23:27:48,2026-09-30 23:33:52,249,90,1.25,11.42,0.00,0.00,16.17,0,0 days 00:06:04,6.066667
6767270,2026-09-30 23:41:50,2026-09-30 23:56:32,170,74,3.84,20.21,0.00,0.00,24.96,0,0 days 00:14:42,14.700000
6767271,2026-09-30 23:15:30,2026-09-30 23:23:00,234,158,1.19,13.66,0.00,0.00,18.41,0,0 days 00:07:30,7.500000
6767272,2026-09-30 23:56:47,2026-10-01 00:05:06,170,48,1.05,15.40,0.00,0.00,20.15,0,0 days 00:08:19,8.316667


tpep_pickup_datetime      datetime64[ns]
tpep_dropoff_datetime     datetime64[ns]
PULocationID                       int64
DOLocationID                       int64
trip_distance                    float64
fare_amount                      float64
tip_amount                       float64
tolls_amount                     float64
total_amount                     float64
payment_type                       int64
duration                 timedelta64[ns]
duration_minutes                 float64
dtype: object

In [ ]:
### Profitability Index
# (Fare Amount + Tip Amount - Tolls) / Trip Duration (Minutes)

trips_df["profitability_index"] = (trips_df["fare_amount"] + trips_df["tip_amount"] - trips_df["tolls_amount"])/ trips_df["duration_minutes"]

display(trips_df)
trips_df.dtypes

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount,payment_type,duration,duration_minutes,profitability_index
0,2026-08-01 00:03:04,2026-08-01 00:07:51,140,262,0.60,6.50,2.30,0.00,13.80,1,0 days 00:04:47,4.783333,1.839721
1,2026-08-01 00:38:45,2026-08-01 00:44:02,140,262,0.70,6.50,3.45,0.00,14.95,1,0 days 00:05:17,5.283333,1.883281
2,2026-08-01 00:54:51,2026-08-01 01:01:07,140,263,1.10,7.90,2.60,0.00,15.50,1,0 days 00:06:16,6.266667,1.675532
3,2026-08-01 00:27:35,2026-08-01 00:54:27,164,80,5.24,29.60,8.56,7.46,51.37,1,0 days 00:26:52,26.866667,1.142680
4,2026-08-01 00:39:08,2026-08-01 01:00:23,148,141,3.57,20.50,5.25,0.00,31.50,1,0 days 00:21:15,21.250000,1.211765
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6767269,2026-09-30 23:27:48,2026-09-30 23:33:52,249,90,1.25,11.42,0.00,0.00,16.17,0,0 days 00:06:04,6.066667,1.882418
6767270,2026-09-30 23:41:50,2026-09-30 23:56:32,170,74,3.84,20.21,0.00,0.00,24.96,0,0 days 00:14:42,14.700000,1.374830
6767271,2026-09-30 23:15:30,2026-09-30 23:23:00,234,158,1.19,13.66,0.00,0.00,18.41,0,0 days 00:07:30,7.500000,1.821333
6767272,2026-09-30 23:56:47,2026-10-01 00:05:06,170,48,1.05,15.40,0.00,0.00,20.15,0,0 days 00:08:19,8.316667,1.851703


tpep_pickup_datetime      datetime64[ns]
tpep_dropoff_datetime     datetime64[ns]
PULocationID                       int64
DOLocationID                       int64
trip_distance                    float64
fare_amount                      float64
tip_amount                       float64
tolls_amount                     float64
total_amount                     float64
payment_type                       int64
duration                 timedelta64[ns]
duration_minutes                 float64
profitability_index              float64
dtype: object

In [ ]:
### Tip Propensity Probability
# % of total zone trips where (Tip Amount / Fare Amount) > 0.20

trips_df["tip_percent"] = (trips_df["tip_amount"])/ trips_df["fare_amount"]
trips_df["good_tip"] = (np.where(trips_df["tip_percent"] > 0.2, 1, 0))



,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount,payment_type,duration,duration_minutes,profitability_index,tip_percent,good_tip
0,2026-08-01 00:03:04,2026-08-01 00:07:51,140,262,0.60,6.50,2.30,0.00,13.80,1,0 days 00:04:47,4.783333,1.839721,0.353846,1
1,2026-08-01 00:38:45,2026-08-01 00:44:02,140,262,0.70,6.50,3.45,0.00,14.95,1,0 days 00:05:17,5.283333,1.883281,0.530769,1
2,2026-08-01 00:54:51,2026-08-01 01:01:07,140,263,1.10,7.90,2.60,0.00,15.50,1,0 days 00:06:16,6.266667,1.675532,0.329114,1
3,2026-08-01 00:27:35,2026-08-01 00:54:27,164,80,5.24,29.60,8.56,7.46,51.37,1,0 days 00:26:52,26.866667,1.142680,0.289189,1
4,2026-08-01 00:39:08,2026-08-01 01:00:23,148,141,3.57,20.50,5.25,0.00,31.50,1,0 days 00:21:15,21.250000,1.211765,0.256098,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6767269,2026-09-30 23:27:48,2026-09-30 23:33:52,249,90,1.25,11.42,0.00,0.00,16.17,0,0 days 00:06:04,6.066667,1.882418,0.000000,0
6767270,2026-09-30 23:41:50,2026-09-30 23:56:32,170,74,3.84,20.21,0.00,0.00,24.96,0,0 days 00:14:42,14.700000,1.374830,0.000000,0
6767271,2026-09-30 23:15:30,2026-09-30 23:23:00,234,158,1.19,13.66,0.00,0.00,18.41,0,0 days 00:07:30,7.500000,1.821333,0.000000,0
6767272,2026-09-30 23:56:47,2026-10-01 00:05:06,170,48,1.05,15.40,0.00,0.00,20.15,0,0 days 00:08:19,8.316667,1.851703,0.000000,0


tpep_pickup_datetime      datetime64[ns]
tpep_dropoff_datetime     datetime64[ns]
PULocationID                       int64
DOLocationID                       int64
trip_distance                    float64
fare_amount                      float64
tip_amount                       float64
tolls_amount                     float64
total_amount                     float64
payment_type                       int64
duration                 timedelta64[ns]
duration_minutes                 float64
profitability_index              float64
tip_percent                      float64
good_tip                           int64
dtype: object

In [ ]:
tip_PU_zone = (
    trips_df.groupby("PULocationID")["good_tip"]
    .mean()
    .mul(100)
    .reset_index(name="good_tip_percentage")
)

tip_PU_zone = tip_PU_zone.sort_values(by="good_tip_percentage", ascending=False)


In [ ]:
tip_DO_zone = (
    trips_df.groupby("DOLocationID")["good_tip"]
    .mean()
    .mul(100)
    .reset_index(name="good_tip_percentage")
)

tip_DO_zone = tip_DO_zone.sort_values(by="good_tip_percentage", ascending=False)

In [ ]:
display(tip_PU_zone)
display(tip_DO_zone)

,PULocationID,good_tip_percentage
195,199,83.333333
134,138,72.122231
69,70,65.775105
233,237,62.261612
0,1,62.068966
...,...,...
200,204,0.000000
183,187,0.000000
111,115,0.000000
43,44,0.000000


,DOLocationID,good_tip_percentage
232,236,62.401840
233,237,61.240038
235,239,59.275091
42,43,58.973605
135,138,58.706303
...,...,...
166,169,4.473439
93,94,4.425837
77,78,4.412547
107,110,0.000000
